# 从ratings_Sports_and_Outdoors.csv文件中提取U-I交互图, 5-core后重新编号
- Extracting U-I interactions and performing 5-core, re-indexing
- dataset located at: http://jmcauley.ucsd.edu/data/amazon/links.html, rating only file in "Small" subsets for experimentation

In [1]:
import os, csv
import pandas as pd
import json
import gzip

In [2]:
os.chdir('/home/raozhongtao/MMRec/data')
os.getcwd()
dataset = "beauty"
raw_data_path = f"{dataset}/{dataset}.json.gz"
csv_path = f"{dataset}/{dataset}.csv"

In [ ]:
def parse(path):
  g = gzip.open(path, 'r')
  for l in g:
    yield eval(l)

def parse_json_gz_to_csv(input_path, output_path):
    with gzip.open(input_path, 'rt', encoding='utf-8') as g, open(output_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['userID', 'itemID', 'rating', 'timestamp'])  # Writing the header
        
        for line in g:
            data = json.loads(line)
            try:
                user_id = str(data.get('reviewerID', ''))
                item_id = str(data.get('asin', ''))
                rating = float(data.get('overall'))
                timestamp = int(data.get('unixReviewTime'))
                

                writer.writerow([user_id, item_id, float(rating), int(timestamp)])
            except (ValueError, KeyError) as e:
                print(f"Skipping invalid entry: {data}, Error: {e}")
            

parse_json_gz_to_csv(raw_data_path, csv_path)    

## 先5-core过滤
## 5-core filtering

In [11]:

df = pd.read_csv(csv_path, names=['userID', 'itemID', 'rating', 'timestamp'], header=None, low_memory=False)
print(f'shape: {df.shape}')
df[:5]


shape: (2023071, 4)


,userID,itemID,rating,timestamp
0,userID,itemID,rating,timestamp
1,A39HTATAQ9V7YF,0205616461,5.0,1369699200
2,A3JM6GV9MNOF9X,0558925278,3.0,1355443200
3,A1Z513UWSAAO0F,0558925278,5.0,1404691200
4,A1WMRR494NWEWV,0733001998,4.0,1382572800


In [12]:
k_core = 5
learner_id, course_id, tmstmp_str = 'userID', 'itemID', 'timestamp'

df.dropna(subset=[learner_id, course_id, tmstmp_str], inplace=True)
df.drop_duplicates(subset=[learner_id, course_id, tmstmp_str], inplace=True)
print(f'After dropped: {df.shape}')
df[:3]

After dropped: (2023071, 4)


,userID,itemID,rating,timestamp
0,userID,itemID,rating,timestamp
1,A39HTATAQ9V7YF,0205616461,5.0,1369699200
2,A3JM6GV9MNOF9X,0558925278,3.0,1355443200


In [13]:
from collections import Counter
import numpy as np

min_u_num, min_i_num = 5, 5

def get_illegal_ids_by_inter_num(df, field, max_num=None, min_num=None):
    if field is None:
        return set()
    if max_num is None and min_num is None:
        return set()

    max_num = max_num or np.inf
    min_num = min_num or -1

    ids = df[field].values
    inter_num = Counter(ids)
    ids = {id_ for id_ in inter_num if inter_num[id_] < min_num or inter_num[id_] > max_num}
    print(f'{len(ids)} illegal_ids_by_inter_num, field={field}')

    return ids


def filter_by_k_core(df):
    while True:
        ban_users = get_illegal_ids_by_inter_num(df, field=learner_id, max_num=None, min_num=min_u_num)
        ban_items = get_illegal_ids_by_inter_num(df, field=course_id, max_num=None, min_num=min_i_num)
        if len(ban_users) == 0 and len(ban_items) == 0:
            return

        dropped_inter = pd.Series(False, index=df.index)
        if learner_id:
            dropped_inter |= df[learner_id].isin(ban_users)
        if course_id:
            dropped_inter |= df[course_id].isin(ban_items)
        print(f'{len(dropped_inter)} dropped interactions')
        df.drop(df.index[dropped_inter], inplace=True)



## k-core

In [14]:
filter_by_k_core(df)
print(f'k-core shape: {df.shape}')
print(f'shape after k-core: {df.shape}')
df[:2]

1157898 illegal_ids_by_inter_num, field=userID
181930 illegal_ids_by_inter_num, field=itemID
2023071 dropped interactions
11978 illegal_ids_by_inter_num, field=userID
37920 illegal_ids_by_inter_num, field=itemID
394908 dropped interactions
12670 illegal_ids_by_inter_num, field=userID
2327 illegal_ids_by_inter_num, field=itemID
285290 dropped interactions
1385 illegal_ids_by_inter_num, field=userID
3310 illegal_ids_by_inter_num, field=itemID
235877 dropped interactions
2370 illegal_ids_by_inter_num, field=userID
409 illegal_ids_by_inter_num, field=itemID
219260 dropped interactions
310 illegal_ids_by_inter_num, field=userID
755 illegal_ids_by_inter_num, field=itemID
208668 dropped interactions
649 illegal_ids_by_inter_num, field=userID
104 illegal_ids_by_inter_num, field=itemID
204558 dropped interactions
82 illegal_ids_by_inter_num, field=userID
211 illegal_ids_by_inter_num, field=itemID
201607 dropped interactions
200 illegal_ids_by_inter_num, field=userID
23 illegal_ids_by_inter_num,

,userID,itemID,rating,timestamp
341,A1YJEY40YUW4SE,7806397051,1.0,1391040000
348,A60XNB876KYML,7806397051,3.0,1397779200


## Re-index

In [15]:
df.reset_index(drop=True, inplace=True)

In [16]:

i_mapping_file = 'i_id_mapping.csv'
u_mapping_file = 'u_id_mapping.csv'

splitting = [0.8, 0.1, 0.1]
uid_field, iid_field = learner_id, course_id

uni_users = pd.unique(df[uid_field])
uni_items = pd.unique(df[iid_field])

# start from 0
u_id_map = {k: i for i, k in enumerate(uni_users)}
i_id_map = {k: i for i, k in enumerate(uni_items)}

df[uid_field] = df[uid_field].map(u_id_map)
df[iid_field] = df[iid_field].map(i_id_map)
df[uid_field] = df[uid_field].astype(int)
df[iid_field] = df[iid_field].astype(int)

# dump
rslt_dir = './'
u_df = pd.DataFrame(list(u_id_map.items()), columns=['user_id', 'userID'])
i_df = pd.DataFrame(list(i_id_map.items()), columns=['asin', 'itemID'])

u_df.to_csv(os.path.join(rslt_dir, u_mapping_file), sep='\t', index=False)
i_df.to_csv(os.path.join(rslt_dir, i_mapping_file), sep='\t', index=False)
print(f'mapping dumped...')

mapping dumped...


In [17]:

# =========2. splitting
print(f'splitting ...')
tot_ratio = sum(splitting)
# remove 0.0 in ratios
ratios = [i for i in splitting if i > .0]
ratios = [_ / tot_ratio for _ in ratios]
split_ratios = np.cumsum(ratios)[:-1]

#df[tmstmp_str] = df[tmstmp_str].map(lambda x: datetime.strptime(x, "%Y-%m-%dT%H:%M:%SZ"))
split_ratios

splitting ...


array([0.8, 0.9])

In [18]:
ts_id = 'timestamp'

split_timestamps = list(np.quantile(df[ts_id], split_ratios))
# get df training dataset unique users/items
df_train = df.loc[df[ts_id] < split_timestamps[0]].copy()
df_val = df.loc[(split_timestamps[0] <= df[ts_id]) & (df[ts_id] < split_timestamps[1])].copy()
df_test = df.loc[(split_timestamps[1] <= df[ts_id])].copy()

x_label, rslt_file = 'x_label', 'sports14-indexed.inter'
df_train[x_label] = 0
df_val[x_label] = 1
df_test[x_label] = 2
temp_df = pd.concat([df_train, df_val, df_test])
temp_df = temp_df[[learner_id, course_id, 'rating', ts_id, x_label]]
print(f'columns: {temp_df.columns}')

temp_df.columns = [learner_id, course_id, 'rating', ts_id, x_label]

temp_df.to_csv(os.path.join(rslt_dir, rslt_file), sep='\t', index=False)
temp_df[:5]
#print('done!')

TypeError: unsupported operand type(s) for -: 'str' and 'str'

## Reload

In [11]:
indexed_df = pd.read_csv(rslt_file, sep='\t')
print(f'shape: {indexed_df.shape}')
indexed_df[:4]

shape: (296337, 5)


,userID,itemID,rating,timestamp,x_label
0,1,0,5.0,1328140800,0
1,2,0,4.0,1330387200,0
2,3,0,4.0,1328400000,0
3,4,0,4.0,1366675200,0


In [12]:
u_uni = indexed_df[learner_id].unique()
c_uni = indexed_df[course_id].unique()

print(f'# of unique learners: {len(u_uni)}')
print(f'# of unique courses: {len(c_uni)}')

print('min/max of unique learners: {0}/{1}'.format(min(u_uni), max(u_uni)))
print('min/max of unique courses: {0}/{1}'.format(min(c_uni), max(c_uni)))


# of unique learners: 35598
# of unique courses: 18357
min/max of unique learners: 0/35597
min/max of unique courses: 0/18356
